In [1]:
import torch
import torch.nn.functional as F


loss = torch.tensor([0.5, 0.7, 0.7, 1.2, 0.7, 1.2, 0.7, 1.2, 1.5])
seq_lens = [2, 3, 4]
loss_split = torch.split(loss, seq_lens)
nested_tensor = torch.nested.nested_tensor(list(loss_split))
padded_tensor = torch.nested.to_padded_tensor(nested_tensor, -1)
mask = padded_tensor == -1
mask.sum(dim=0)

/home/fsuser/miniconda3/envs/ncs-stable/lib/python3.11/site-packages/torch/nested/__init__.py:151: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at /opt/conda/conda-bld/pytorch_1697614972865/work/aten/src/ATen/NestedTensorImpl.cpp:178.)
  return _nested.nested_tensor(


tensor([0, 0, 1, 2])

In [ ]:
def tensor_split_per_pos(values_tensor: torch.Tensor, seq_lens: List[int], max_len: int):
  seq_size = sum(seq_lens)
  non_padded_values_tensor = values_tensor[:seq_size]
  masks = torch.nested.nested_tensor([torch.ones(l, device=values_tensor.device) for l in seq_lens])
  masks = torch.nested.to_padded_tensor(masks, (len(seq_lens), max_len))
  divisor = torch.sum(masks, dim=0)
  seq_split_tensor = torch.split(non_padded_values_tensor, seq_lens, dim=0)
  seq_split_tensor = torch.nested.nested_tensor(seq_split_tensor)
  seq_split_tensor = torch.nested.to_padded_tensor(seq_split_tensor, (len(seq_lens), max_len))
  seq_split_acc = torch.sum(seq_split_tensor, dim=0)
  return seq_split_acc / divisor

In [11]:
loss      = torch.tensor([0.5, 0.7, 0.7, 1.2, 0.7, 1.2, 0.7, 1.2, 1.5])
positions = torch.tensor([  3,   4,   5,   6,   0,   1,   0,   3,   4])

agg_vals = torch.zeros(positions.max() + 1)
agg_vals.scatter_add_(0, positions, loss)
pos_cnt = torch.bincount(positions)


print(agg_vals)
print(pos_cnt)

tensor([1.4000, 1.2000, 0.0000, 1.7000, 2.2000, 0.7000, 1.2000])
tensor([2, 1, 0, 2, 2, 1, 1])


In [25]:
loss      = torch.tensor([0.5, 0.7, 0.7, 1.2, 0.7, 1.2, 0.7, 1.2, 1.5])
ds_mask = torch.tensor([  3,   5,   5,   6,   6,   2,   2,   3,   23])
# ds_mask = torch.tensor([  3,   4,   5,   6,   0,   1,   0,   3,   4])

# Initialize tensors for sum and count
sum_vals = torch.zeros(ds_mask.max() + 1)
counts = torch.zeros(ds_mask.max() + 1)

# Use torch.scatter_add() to sum values based on mask positions
sum_vals.scatter_add_(0, ds_mask, loss)

# Count occurrences of each position in the mask
counts.scatter_add_(0, ds_mask, torch.ones_like(loss))

means = sum_vals/counts
tensor_dict = {idx: val.item() for idx, val in enumerate(means) if not torch.isnan(val)}


print(ds_mask)
print(sum_vals)
print(counts)
print(means)
from pprint import pprint
pprint(tensor_dict)

tensor([ 3,  5,  5,  6,  6,  2,  2,  3, 23])
tensor([0.0000, 0.0000, 1.9000, 1.7000, 0.0000, 1.4000, 1.9000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.5000])
tensor([0., 0., 2., 2., 0., 2., 2., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 1.])
tensor([   nan,    nan, 0.9500, 0.8500,    nan, 0.7000, 0.9500,    nan,    nan,
           nan,    nan,    nan,    nan,    nan,    nan,    nan,    nan,    nan,
           nan,    nan,    nan,    nan,    nan, 1.5000])
{2: 0.9500000476837158,
 3: 0.8500000238418579,
 5: 0.699999988079071,
 6: 0.9500000476837158,
 23: 1.5}


In [ ]:

                   0 1 2    0 1 2
t = torch.tensor([[0,1,3], [3,1,0]])




